In [ ]:
# Upload the dataset
from google.colab import files

uploaded = files.upload()

In [ ]:
# Import the required libraries
!pip install ctgan

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_auc_score, recall_score
from tabulate import tabulate
import random, numpy as np, torch
from ctgan import CTGAN
import time

In [ ]:
# Reproducibility and GPU check

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device     :", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU is active. In Colab: Runtime > Change runtime type > Hardware accelerator = GPU (T4/L4), "
        "then restart the session and run the notebook from the first cell."
    )

In [ ]:
# Load the CSV file into a pandas DataFrame
file_name = 'df_train.csv'
df = pd.read_csv(file_name)

df.info()

## Option A - discrete-value schema

In [ ]:
# OPTION A - DISCRETE-VALUE SCHEMA
import numpy as np
import pandas as pd

SCHEMA_TARGET_COL   = "Target"
MAX_DISCRETE_LEVELS = 12          # at most this many unique values => discrete
ONEHOT_PREFIXES     = ["Urine Test_"]

def build_schema(df_reference, target_col=SCHEMA_TARGET_COL):
    """Describe every feature column of the real training data."""
    schema = {"discrete": {}, "continuous": {}, "onehot_groups": {}}
    features = [c for c in df_reference.columns if c != target_col]

    for prefix in ONEHOT_PREFIXES:
        members = sorted(c for c in features if c.startswith(prefix))
        if members:
            schema["onehot_groups"][prefix] = members

    grouped = {c for m in schema["onehot_groups"].values() for c in m}

    for col in features:
        if col in grouped:
            continue
        levels = np.unique(df_reference[col].dropna().to_numpy(dtype=float))
        if len(levels) <= MAX_DISCRETE_LEVELS:
            schema["discrete"][col] = np.sort(levels)
        else:
            schema["continuous"][col] = (
                float(df_reference[col].min()),
                float(df_reference[col].max()),
            )
    return schema

def apply_schema(df_synth, schema, target_col=SCHEMA_TARGET_COL):
    """Return a copy of df_synth in which every value is legal."""
    out = df_synth.copy()

    # 1. snap each discrete column to its nearest legal level
    for col, levels in schema["discrete"].items():
        if col not in out.columns:
            continue
        values = out[col].to_numpy(dtype=float)
        idx = np.abs(values[:, None] - levels[None, :]).argmin(axis=1)
        out[col] = levels[idx]

    # 2. reduce each one-hot group to a single 1 (arg-max wins)
    for members in schema["onehot_groups"].values():
        members = [c for c in members if c in out.columns]
        if not members:
            continue
        block = out[members].to_numpy(dtype=float)
        hard = np.zeros_like(block)
        hard[np.arange(len(block)), block.argmax(axis=1)] = 1.0
        out[members] = hard

    # 3. keep continuous columns inside the range seen in the real data
    for col, (lo, hi) in schema["continuous"].items():
        if col in out.columns:
            out[col] = out[col].clip(lo, hi)

    return out

def validity_report(df, schema, target_col=SCHEMA_TARGET_COL, label=""):
    """Percentage of rows holding a legal value, per discrete column."""
    rows = []
    for col, levels in schema["discrete"].items():
        if col not in df.columns:
            continue
        ok = np.isin(df[col].to_numpy(dtype=float), levels).mean() * 100.0
        rows.append((col, "discrete", len(levels), round(ok, 2)))

    for prefix, members in schema["onehot_groups"].items():
        members = [c for c in members if c in df.columns]
        if not members:
            continue
        block = df[members].to_numpy(dtype=float)
        ok = (np.isin(block, [0.0, 1.0]).all(axis=1)
              & (block.sum(axis=1) == 1.0)).mean() * 100.0
        rows.append((prefix + "*", "one-hot", len(members), round(ok, 2)))

    rep = pd.DataFrame(rows, columns=["Column", "Type", "Levels", "Valid (%)"])
    rep = rep.sort_values("Valid (%)").reset_index(drop=True)
    print("\n--- Validity report " + str(label) + " ---")
    print("Columns fully valid : {} / {}".format((rep["Valid (%)"] == 100).sum(), len(rep)))
    print("Mean validity       : {:.2f}%".format(rep["Valid (%)"].mean()))
    display(rep)
    return rep

# Build the schema from the real training data
_df_reference = pd.read_csv("df_train.csv")
SCHEMA = build_schema(_df_reference)

print("Discrete columns   :", len(SCHEMA["discrete"]))
print("One-hot groups     :", len(SCHEMA["onehot_groups"]))
print("Continuous columns :", len(SCHEMA["continuous"]))
print("\nDiscrete columns and their legal levels:")
for _c, _l in SCHEMA["discrete"].items():
    print("  {:<32} {}".format(_c, np.round(_l, 4).tolist()))

In [ ]:
# Class distribution of the current DataFrame
counts = df["Target"].value_counts().sort_index()

plt.figure()
bars = plt.bar(counts.index.astype(str), counts.values)

for i, value in enumerate(counts.values):
    plt.text(i, value, str(value), ha='center', va='bottom')

plt.xlabel("Target")
plt.ylabel("Number of records")
plt.title("Class distribution of the target")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
pd.set_option('display.max_columns', None)
df.head()

In [ ]:
# 1. Load the training data

df = pd.read_csv('df_train.csv')

target_col = 'Target'

print("Training set shape:")
print(df.shape)

print("\nDataset columns:")
print(df.columns.tolist())

# 2. Feature grouping

# ORDINAL FEATURES

ordinal_columns = [
    'Physical Activity', 'Socioeconomic Factors', 'Alcohol Consumption'
]

# BINARY FEATURES
binary_columns = [
    'Dietary Habits', 'Ethnicity', 'Liver Function Tests',
    'Genetic Markers', 'Autoantibodies', 'Family History',
    'Environmental Factors', 'Smoking Status',
    'Glucose Tolerance Test', 'History of PCOS',
    'Previous Gestational Diabetes', 'Pregnancy History',
    'Cystic Fibrosis Diagnosis', 'Steroid Use History',
    'Genetic Testing', 'Early Onset Symptoms'
]

# NOMINAL / ONE-HOT FEATURES

nominal_columns = [
    col for col in df.columns if col.startswith('Urine Test_')
]

# CONTINUOUS FEATURES

continuous_columns = [
    'Age', 'BMI', 'Insulin Levels', 'Blood Pressure',
    'Cholesterol Levels', 'Waist Circumference',
    'Blood Glucose Levels', 'Weight Gain During Pregnancy',
    'Pancreatic Health', 'Pulmonary Function',
    'Neurological Assessments', 'Digestive Enzyme Levels',
    'Birth Weight'
]

# 3. Column validation

print("\n=== FEATURE GROUPING ===")

print("\nOrdinal Features:")
print(ordinal_columns)

print("\nBinary Features:")
print(binary_columns)

print("\nNominal / One-Hot Features:")
print(nominal_columns)

print("\nContinuous Features:")
print(continuous_columns)

# 4. Report any missing column

expected_columns = (
    ordinal_columns + binary_columns + continuous_columns + nominal_columns
)

missing_columns = [
    col for col in expected_columns
    if col not in df.columns
]

if missing_columns:
    print("\nColumns not found:")
    for col in missing_columns:
        print("-", col)
else:
    print("\nAll expected features were found.")

# 5. Target distribution

if target_col in df.columns:
    print("\n=== TRAINING TARGET DISTRIBUTION ===")

    target_distribution = (
        df[target_col].value_counts().sort_index()
    )

    display(target_distribution.to_frame('Count'))

# 6. Data types

print("\n=== DATA TYPES ===")

display(
    df[expected_columns + [target_col]].dtypes.to_frame('Data Type')
)

# 7. Ordinal value check

print("\n=== ORDINAL VALUE CHECK ===")

for col in ordinal_columns:
    if col in df.columns:
        print(f"{col}: {sorted(df[col].dropna().unique())}")

# 8. Binary value check

print("\n=== BINARY VALUE CHECK ===")

for col in binary_columns:
    if col in df.columns:
        print(f"{col}: {sorted(df[col].dropna().unique())}")

# 9. One-hot feature check

print("\n=== ONE-HOT FEATURE CHECK ===")

for col in nominal_columns:
    print(f"{col}: {sorted(df[col].dropna().unique())}")

## Checkpointing — mount Drive

In [ ]:
# CHECKPOINTING  —  survive a Colab disconnect
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_ROOT = '/content/drive/MyDrive/diabetes_gan_checkpoints'
except Exception as e:
    print('Not running on Colab, or Drive was refused:', e)
    print('Falling back to a local folder — this does NOT survive a disconnect.')
    CKPT_ROOT = './checkpoints'

# False retrains every class from scratch and overwrites the checkpoints
RESUME = True

print('Checkpoint root :', CKPT_ROOT)
print('Resume enabled  :', RESUME)

In [ ]:
# Configuration
TARGET_COLUMN    = "Target"
TARGET_PER_CLASS = 6000

CTGAN_EPOCHS = 300      # scenarios: 100, 300, 500
CTGAN_BATCH  = 64       # must be a multiple of PAC
CTGAN_PAC    = 1

USE_GPU = True          # run CTGAN on the GPU

# Checkpoint folder — one per (epochs, target, seed) combination so
CKPT_DIR = os.path.join(
    CKPT_ROOT, f'ctgan_epoch{CTGAN_EPOCHS}_augment{TARGET_PER_CLASS}_seed{SEED}')
os.makedirs(CKPT_DIR, exist_ok=True)
print('Checkpoint folder:', CKPT_DIR)

def class_ckpt_path(label):
    safe = str(label).replace('/', '_').replace(' ', '_')
    return os.path.join(CKPT_DIR, f'class_{safe}.csv')

def checkpoint_status(df):
    """What is already done, and what is still missing."""
    rows = []
    for label in sorted(df[TARGET_COLUMN].unique()):
        have = len(df[df[TARGET_COLUMN] == label])
        need = max(TARGET_PER_CLASS - have, 0)
        path = class_ckpt_path(label)
        cached = os.path.exists(path)
        n_cached = len(pd.read_csv(path)) if cached else 0
        rows.append({
            'Class': label, 'Original': have, 'Synthetic needed': need,
            'Checkpoint': 'yes' if cached else '-', 'Rows stored': n_cached,
            'State': ('not needed' if need == 0
                      else 'done' if n_cached == need
                      else 'partial/invalid' if cached else 'PENDING'),
        })
    status = pd.DataFrame(rows)
    pending = (status['State'] == 'PENDING').sum()
    print(f'\n{len(status) - pending - (status["State"] == "not needed").sum()}'
          f' of {(status["State"] != "not needed").sum()} classes already trained; '
          f'{pending} still to run.')
    return status

# Per-class CTGAN training, with resume
def train_ctgan_per_class(df, discrete_columns):

    all_data = []
    synthetic_parts = []   # kept so the synthetic rows can be exported alone
    loaded, trained = [], []

    for target_class in sorted(df[TARGET_COLUMN].unique()):

        print(f"\nProcessing class: {target_class}")

        df_class = df[df[TARGET_COLUMN] == target_class].copy()
        current_count = len(df_class)
        print(f"Original samples: {current_count}")

        all_data.append(df_class)

        num_to_generate = TARGET_PER_CLASS - current_count
        if num_to_generate <= 0:
            print("Class already large enough, augmentation skipped")
            continue

        path = class_ckpt_path(target_class)

        # ---- resume from an earlier session ----------------------
        if RESUME and os.path.exists(path):
            cached = pd.read_csv(path)
            same_cols = list(cached.columns) == list(df.columns)
            same_size = len(cached) == num_to_generate

            if same_cols and same_size:
                print(f"   Loaded from checkpoint: {len(cached):,} rows")
                all_data.append(cached)
                synthetic_parts.append(cached)
                loaded.append(target_class)
                continue

            print("   Checkpoint does not match this configuration "
                  f"(rows {len(cached)} vs {num_to_generate}, "
                  f"columns {'ok' if same_cols else 'differ'}) — retraining.")

        # ---- train ----------------------------------------------
        ctgan = CTGAN(
            epochs=CTGAN_EPOCHS,
            batch_size=CTGAN_BATCH,
            pac=CTGAN_PAC,
            verbose=True,
            enable_gpu=USE_GPU
        )

        # Per-class seed: deterministic, yet different for each class.
        try:
            ctgan.set_random_state(SEED + int(target_class))
        except Exception as e:
            print(f"   set_random_state skipped: {e}")

        t0 = time.time()
        ctgan.fit(df_class, discrete_columns)
        elapsed = time.time() - t0

        # Verify that the model is really placed on the GPU
        try:
            on_cuda = next(ctgan._generator.parameters()).is_cuda
            print(f"   Generator on CUDA : {on_cuda}")
            assert on_cuda, "CTGAN is still running on the CPU."
        except AttributeError:
            print("   (_generator attribute unavailable, verification skipped)")

        print(f"   Training time     : {elapsed/60:.2f} minutes")

        synthetic = ctgan.sample(num_to_generate)
        synthetic[TARGET_COLUMN] = target_class
        synthetic = synthetic[df.columns]       # keep the column order fixed

        # ---- write the checkpoint BEFORE moving on ---------------
        synthetic.to_csv(path, index=False)
        print(f"Synthetic samples generated: {len(synthetic)}")
        print(f"   Checkpoint saved  : {path}")

        all_data.append(synthetic)
        synthetic_parts.append(synthetic)
        trained.append(target_class)

    print('\n' + '=' * 56)
    print(f'Loaded from checkpoint : {len(loaded)} classes  {loaded}')
    print(f'Trained in this session: {len(trained)} classes  {trained}')
    print('=' * 56)

    combined = pd.concat(all_data, ignore_index=True)
    synthetic_only = (pd.concat(synthetic_parts, ignore_index=True)
                      if synthetic_parts else pd.DataFrame(columns=combined.columns))
    return combined, synthetic_only

In [ ]:
# Discrete columns declared to CTGAN

# Binary, ordinal, one-hot nominal and target columns
discrete_columns = (
    binary_columns
    + ordinal_columns
    + nominal_columns
    + ['Target']
)

# Keep only the columns present in the dataset
discrete_columns = [
    col for col in discrete_columns
    if col in df.columns
]

print("=== DISCRETE COLUMNS FOR CTGAN ===")
for col in discrete_columns:
    print("-", col)

print("\nNumber of discrete columns:", len(discrete_columns))

## Checkpoint status

In [ ]:
# What is already done?
status = checkpoint_status(df)
display(status)

In [ ]:
# Train CTGAN class by class

df_balanced, df_synthetic = train_ctgan_per_class(
    df,
    discrete_columns
)

print("Synthetic rows generated:", f"{len(df_synthetic):,}")

print("\n=== CLASS DISTRIBUTION AFTER AUGMENTATION ===")
print(
    df_balanced[TARGET_COLUMN]
    .value_counts()
    .sort_index()
)

# Validity check

In [ ]:
# OPTION A - VALIDITY CHECK FOR CTGAN
rep_before = validity_report(df_balanced, SCHEMA, TARGET_COLUMN, "CTGAN BEFORE correction")

df_balanced = apply_schema(df_balanced, SCHEMA, TARGET_COLUMN)
df_synthetic = apply_schema(df_synthetic, SCHEMA, TARGET_COLUMN)

rep_after = validity_report(df_balanced, SCHEMA, TARGET_COLUMN, "CTGAN AFTER correction")

comparison = rep_before.merge(
    rep_after, on=["Column", "Type", "Levels"], suffixes=(" before", " after")
)
display(comparison)

print("\nRows in the balanced dataset: {:,}".format(len(df_balanced)))

In [ ]:
df_balanced["Target"].value_counts()

In [ ]:
# STANDARDISED EXPORT
MODEL_TAG = "ctgan_perclass"
DEVICE_TAG = "gpu" if USE_GPU else "cpu"
STEM = f"ctgan_perclass_{DEVICE_TAG}_epoch{CTGAN_EPOCHS}_augment{TARGET_PER_CLASS}_seed{SEED}"

COMBINED_FILE = f"{STEM}.csv"
SYNTHETIC_ONLY_FILE = f"{STEM}_synthetic_only.csv"

# ---- checks before anything is written -------------------------
assert len(df) + len(df_synthetic) == len(df_balanced), (
    "Combined frame is not original + synthetic: "
    f"{len(df)} + {len(df_synthetic)} != {len(df_balanced)}")
assert list(df_balanced.columns) == list(df_synthetic.columns), \
    "Column order differs between the combined and the synthetic frame."
assert df_balanced.isna().sum().sum() == 0, "NaN found in the combined frame."
assert df_synthetic.isna().sum().sum() == 0, "NaN found in the synthetic frame."

counts = df_balanced[TARGET_COLUMN].value_counts().sort_index()

print("Original training rows :", f"{len(df):,}")
print("Synthetic rows         :", f"{len(df_synthetic):,}")
print("Combined rows          :", f"{len(df_balanced):,}")
print("Per-class counts       :", counts.tolist())

if counts.nunique() == 1:
    print("All classes balanced at", counts.iloc[0])
else:
    print("WARNING - the classes are not balanced.")

df_balanced.to_csv(COMBINED_FILE, index=False)
df_synthetic.to_csv(SYNTHETIC_ONLY_FILE, index=False)

print("\nSaved:")
print("  ", COMBINED_FILE)
print("       scenario 2 - df_train + synthetic")
print("  ", SYNTHETIC_ONLY_FILE)
print("       scenario 3 - synthetic only, and the input for notebook 06")

files.download(COMBINED_FILE)
files.download(SYNTHETIC_ONLY_FILE)